# Mode of the legs of transit tours (multinomial logit; not in the article)

A model of DAYMAP (Daily Activity Model for Agent Plans), the activity-based model of the MILES platform, built on the ABIT code base.

## What this model answers

On a tour whose main mode is transit, which mode is each leg? The model compares transit, walk, car passenger, car driver and bike by leg position, distance and the person.

The fewer-cars-than-drivers term is kept: the car allocation runs before the leg choice but covers only part of the effect, and the term stays significant under the free-car rule (car driver only when a household car is free over the tour window).

| | |
|---|---|
| Choice | The mode of a leg |
| Alternatives | transit, walk, car passenger, car driver, bike (as available) |
| Decision makers | Persons on transit tours |
| Data | Regional Travel Survey 2017/18 (Baltimore and Washington regions), estimation half |

## How to run

- Kernel: *Python (biogeme)* (`~/.venvs/biogeme`, Biogeme 3.3). Then *Run All*.
- `RUN_DATA_PREP`: rebuild the records and tables in a scratch copy and compare them with the shipped ones.
- `RUN_CHECK`: log-likelihood at the reference coefficients, no estimation.
- `RUN_ESTIMATION`: run the Biogeme estimation (off by default).
- `RUN_HOLDOUT`: log-likelihood of the hold-out half.

## Contents

This notebook goes from the survey records to the estimates of this model:

1. Raw data: survey files, estimation and hold-out halves, the records of this model
2. Data preparation: filters from the survey to the records (records kept and dropped), rebuild check against the shipped tables
3. Choice set and estimation data
4. Descriptive statistics and figures; 4.1 every explanatory variable of the model in the data
5. Model specification (the folder's `model/estimate_transit_leg_mode.py`, unchanged)
6. Correlation of the explanatory variables (heatmaps, VIF)
7. Log-likelihood at the reference coefficients
8. Biogeme estimation
9. Results: article format, what the coefficients say, fit, comparison with the article, correlation of the estimates
10. Hold-out sample
11. Glossary

## Model at a glance

**1. Who decides.** Persons on transit tours. Data: Regional Travel Survey 2017/18 (Baltimore and Washington regions), estimation half; 6,773 legs.

**2. What is chosen (dependent variable).** The mode of a leg. Alternatives: transit, walk, car passenger, car driver, bike (as available).

**3. Explanatory variables** (signs of the article estimates; terms specific to an alternative or purpose are listed per series):

| Group | Variable | Sign |
|---|---|---|
| Constants and scales | Constant | Walk: +, Car passenger: −, Car driver: +, Bike: − |
| Person, household and destination | First leg | Walk: −, Car passenger: −, Car driver: + |
| Person, household and destination | Last leg | Walk: −, Car passenger: − |
| Person, household and destination | Leg at main activity | Walk: −, Car passenger: −, Car driver: − |
| Person, household and destination | Km | Walk: −, Bike: − |
| Person, household and destination | Ln 1 plus km | Car passenger: −, Car driver: − |
| Person, household and destination | Transit time 10min | Transit: − |
| Person, household and destination | No transit path | Transit: − |
| Person, household and destination | Previous leg car driver | Car driver: + |
| Person, household and destination | First leg car driver on last leg | Car driver: + |
| Person, household and destination | Previous leg car passenger | Car passenger: + |
| Person, household and destination | Previous leg walk | Walk: + |
| Person, household and destination | No car | Car passenger: − |
| Person, household and destination | Fewer cars than drivers | Car driver: − |
| Person, household and destination | Age under 18 | Car passenger: + |
| Person, household and destination | Age 65 plus | Car passenger: − |
| Person, household and destination | Driving licence | Car passenger: − |
| Person, household and destination | Work tour | Walk: +, Car passenger: +, Car driver: + |
| Person, household and destination | School tour | Car passenger: + |
| Person, household and destination | Peak departure | Car passenger: −, Walk: − |
| Person, household and destination | Home in washington region | Walk: +, Car driver: − |

**4. How person and household traits enter.** Traits enter with one coefficient per alternative, against the reference alternative whose utility is 0.

**5. How to read a coefficient.** Positive makes the alternative more likely, negative less likely; exp(coefficient) is the factor by which the odds change for one unit more.

In [ ]:
# switches (see How to run) and set-up: imports, folders, helper modules
RUN_DATA_PREP  = True    # rebuild the records and estimation tables in a scratch copy and compare with the shipped tables (about 2 min)
RUN_CHECK      = True    # log-likelihood at the reference coefficients
RUN_ESTIMATION = False   # Biogeme estimation, written to output/ (False: results are read from output/ if there are any)
RUN_HOLDOUT    = True    # log-likelihood of the hold-out half at the reference coefficients

import os, sys, json, glob, inspect, importlib, time, re
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from biogeme.results_processing import EstimationResults

FOLDER = os.path.abspath("")                       # this model folder (VS Code starts the notebook here)
sys.path.insert(0, os.path.join(FOLDER, "..", "..", "tools")); import nbtools as nt; import nb_reader as rd; import figs as fg; import abit_figs as af
from IPython.display import Markdown
sys.path.insert(0, os.path.join(FOLDER, "model"))
RUN = json.load(open(os.path.join(FOLDER, "RUN.json")))
OUT = os.path.join(FOLDER, "output"); os.makedirs(OUT, exist_ok=True)
NAME = "transit_leg_mode"; MODEL_FOLDER = "20_transit_leg_mode"
pd.set_option("display.max_rows", 200, "display.max_columns", 60, "display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("model folder:", os.path.basename(FOLDER))

## 1. Raw data

*Where the records come from. Look for: the files and the record counts.*

### 1.1 The 2017/18 Regional Travel Survey

All DAYMAP models are estimated on the 2017/18 RTS of the Baltimore and Washington regions (weekday travel day). Three files
are shipped in `../00_shared_data/rts_2017_18/raw`:

- `RTS_raw_flat_data.csv.gz`: the survey as received, one row per trip with its household, person and vehicle fields;
- `rts_trips_clean.csv.gz`: the trips after the survey cleaning (one row per trip, departure / arrival in minutes);
- `rts_tours.csv.gz`: home-based tours, work-based subtours and open tours built from the clean trips.

The cleaning script of the last two files is not shipped; the cleaned files are the starting point of the preparation here.
Race and ethnicity fields are not read.

In [ ]:
# 1.1 The 2017/18 Regional Travel Survey
flat, trips, tours = nt.rts_raw()
split = pd.read_csv(os.path.join(nt.ABIT_SHARED, "rts_2017_18", "split", "rts_split.csv"))
display(nt.rts_counts(flat, trips, tours, split))
show = ["tripid", "household_id", "person_id", "tripno", "o_activity", "d_activity", "dep_min", "arr_min", "travel_mode",
        "distance", "hhsize", "numvehicle", "hh_income_broad", "age", "employment_status", "student_status", "wttrdfin"]
trips[[c for c in show if c in trips]].head()

### 1.2 Estimation and hold-out halves

The households are split in two halves (`make_rts_split.py`): strata of home county x household size (1 to 5+), a seeded
shuffle inside each stratum (seed 2026), then alternate assignment. Every model is estimated on the estimation half; the
hold-out (validation) half is the other half of the same survey and is not read by the estimation. With `RUN_DATA_PREP` the
split is rebuilt into a scratch folder and compared with the shipped `rts_split.csv`.

In [ ]:
# 1.2 Estimation and hold-out halves
REBUILD = os.path.join(OUT, "rebuild_check")            # scratch folder: provided files are never overwritten
if RUN_DATA_PREP:
    rc = nt.run_quiet(f"{sys.executable} make_rts_split.py", cwd=nt.ABIT_PREP, env={"ABIT_REBUILT": REBUILD})
    display(nt.compare_frames(pd.read_csv(os.path.join(REBUILD, "rts_2017_18", "split", "rts_split.csv")), split,
                              key=["household_id"], name="rts_split.csv"))
else:
    print("skipped (RUN_DATA_PREP = False)")
hh = flat.drop_duplicates("household_id").set_index("household_id")
hh["half"] = split.set_index("household_id")["half"]
t = pd.crosstab(hh["hhsize"].clip(upper=5).rename("household size (5 = 5+)"), hh["half"])
t.loc["all"] = t.sum()
t

### 1.3 Records of this model

`input/code/estimate_transit_legs.py` (function `legs`) cuts every transit tour of `tours_<half>.parquet` (tour mode transit
with walk or drive access) into its linked trips (legs): consecutive survey trips joined at a change-of-mode activity become
one leg with the highest mode used. `tlegs_<half>.parquet` holds one row per leg.

In [ ]:
# 1.3 Records of this model
CODE = os.path.join(FOLDER, "input", "code"); RAW = os.path.join(FOLDER, "input", "data", "raw")
HALVES = ["estimation", "validation"]
TOURS = {h: pd.read_parquet(os.path.join(RAW, f"tours_{h}.parquet")) for h in HALVES}
LEGS = {h: pd.read_parquet(os.path.join(RAW, f"tlegs_{h}.parquet")) for h in HALVES}
display(pd.DataFrame({h: {"tours": len(TOURS[h]), "transit tours": int(TOURS[h].choice.isin(["TW", "TD"]).sum()),
                          "legs of transit tours": len(LEGS[h])} for h in HALVES}))
LEGS["estimation"][["tour", "person_id", "q", "n", "first", "last", "main", "mode", "merged", "outing", "oz", "dz", "dep", "tour_choice", "purpose", "w"]].head(8)

## 2. Data preparation

*From the raw records to the estimation sample. Look for: the records kept and dropped at each filter. With RUN_DATA_PREP the rebuilt tables must equal the shipped ones.*

### 2.1 From transit tours to the estimation table

`features` drops the legs of work-based outings (generated by the outing model), legs with an unknown zone or mode, and
legs whose chosen mode is not available (walk up to 8 km, bike up to 30 km, car driver for licence holders 16+ with a car).
`free_car.py` then applies DAYMAP's free-car rule: car driver needs a household car that is free over the whole tour window
(the household's car-driver tours overlapping the window are fewer than its cars at every instant); legs whose observed mode
becomes unavailable are dropped.

In [ ]:
# Data preparation
sys.path.insert(0, CODE)
import estimate_transit_legs as E
import free_car as FC
steps, X_ = {}, {}
for h in HALVES:
    L = LEGS[h]
    s = [("transit tours", int(TOURS[h].choice.isin(["TW", "TD"]).sum())), ("legs of the transit tours", len(L))]
    s.append(("not a leg of a work-based outing", int((~L.outing).sum())))
    l2 = L[~L.outing]; s.append(("zones and mode known", int(((l2.o >= 0) & (l2.d >= 0) & l2["mode"].notna()).sum())))
    X0 = E.features(L); s.append(("chosen mode available (licence, age, household car)", int(X0.chosen_av.sum())))
    X = FC.apply(X0, h); X_[h] = X
    s.append(("chosen mode available with a free household car (estimation table)", int(X.chosen_av.sum())))
    steps[h] = s
display(nt.funnel2(steps, {"not a leg of a work-based outing": "legs of outings from the workplace",
                           "zones and mode known": "zone outside the zone system or unknown mode",
                           "chosen mode available (licence, age, household car)": "chosen mode not available",
                           "chosen mode available with a free household car (estimation table)": "car driver chosen but no household car free over the tour window"}))
pd.concat({h: X_[h][~X_[h].chosen_av]["mode"].value_counts() for h in HALVES}, axis=1).rename_axis("chosen mode of the dropped legs")

### 2.2 Rebuild check

The shared `prep_tours.py` rebuilds the tours into a scratch folder; a scratch copy of `input/code` gets these tours only, so `legs` rebuilds the leg files, and `build_estimation_tables.py` the estimation tables; all are compared with the shipped files.

In [ ]:
# 2.2 Rebuild check
REBUILD = os.path.join(OUT, "rebuild_check")

In [ ]:
# 2.2 Rebuild check
if RUN_DATA_PREP:
    t0 = time.time()
    SCR = nt.scratch_tree(FOLDER, raw_files=[])
    SCODE = os.path.join(SCR, "input", "code")

    nt.run(f"{sys.executable} prep_tours.py", cwd=nt.ABIT_PREP, env={"ABIT_REBUILT": REBUILD}, quiet=True)
    import shutil
    for h in HALVES:
        shutil.copy2(os.path.join(REBUILD, "mode_choice_work", f"tours_{h}.parquet"), os.path.join(SCR, "input", "data", "raw"))


    nt.run("python3 build_estimation_tables.py", cwd=SCODE)
    res = pd.concat([nt.compare_trees(os.path.join(SCR, "input", "data", "raw"), RAW),
                     nt.compare_trees(os.path.join(SCR, "input", "data", "estimation"), os.path.join(FOLDER, "input", "data", "estimation")).assign(half="estimation"),
                     nt.compare_trees(os.path.join(SCR, "input", "data", "holdout"), os.path.join(FOLDER, "input", "data", "holdout")).assign(half="hold-out")],
                    ignore_index=True)
    nt.drop_scratch(SCR)
    print(f"rebuild and comparison: {(time.time() - t0) / 60:.1f} min")
    display(res)
    nt.rebuild_verdict(res, glob.glob(os.path.join(FOLDER, "model", "estimate_*.py"))[0])
else:
    print("skipped (RUN_DATA_PREP = False): the shipped tables are used")

## 3. Choice set and estimation table

*The table the model reads. Look for: the columns used and what they mean (data dictionary).*

Five alternatives (code `y`): 1 transit, 2 walk, 3 car passenger, 4 car driver, 5 bike. Transit and car passenger are always
available, walk up to 8 km, bike up to 30 km, car driver for licence holders aged 16+ in a household with a car that is free
over the tour window.

In [ ]:
# Choice set and estimation table
LAB = {1: "transit", 2: "walk", 3: "car passenger", 4: "car driver", 5: "bike"}
EST_T = pd.read_parquet(os.path.join(FOLDER, "input", "data", "estimation", "estimation_table.parquet"))
HOLD_T = pd.read_parquet(os.path.join(FOLDER, "input", "data", "holdout", "estimation_table.parquet"))
print(f"estimation table: {EST_T.shape[0]:,} rows x {EST_T.shape[1]} columns; hold-out {HOLD_T.shape[0]:,} rows")
av = [c for c in EST_T.columns if c.startswith("av_")]
t = pd.DataFrame({"available (% of legs)": (100 * (EST_T[av] > 0).mean()).round(1)}); t.index = [c[3:] for c in av]
display(t.T); display(EST_T.head())
pd.read_csv(os.path.join(FOLDER, "input", "data", "estimation", "data_dictionary.csv"))

## 4. Descriptive statistics

*What the sample looks like. Look for: plausible shares and maps, and the chosen alternatives against the available ones.*

### Observed leg modes (unweighted and with the survey trip weight `w`)

In [ ]:
# Leg modes overall and by position in the tour: access and egress legs (first, last) carry the walk and car leg
sh = nt.shares_table(EST_T.y.map(LAB), EST_T.w).reindex([v for v in LAB.values()])
display(sh)
nt.grouped_bars(sh[["share %", "weighted share %"]].rename(columns={"share %": "unweighted", "weighted share %": "survey weight"}), "leg mode", "share of legs (%)", colors=[nt.OBS, nt.LIGHT])
plt.tight_layout(); plt.show()
pos = np.select([EST_T["first"] == 1, EST_T["last"] == 1], ["first leg", "last leg"], "middle leg")
display((100 * pd.crosstab(pos, EST_T.y.map(LAB), normalize="index")).round(1).rename_axis(index="position", columns="leg mode"))

Leg modes overall and by position in the tour: access and egress legs (first, last) carry the walk and car legs.

In [ ]:
# Leg distance by mode, tour segment, legs per tour, and the person and leg variables with the transit share amo
fig, ax = nt.panels(3, ncol=3, w=4.4, h=2.9)
labs = [l for m, l in LAB.items() if (EST_T.y == m).any()]
data = [EST_T.km[EST_T.y == m].clip(lower=0.2) for m, l in LAB.items() if (EST_T.y == m).any()]
ax[0].boxplot(data, vert=False, tick_labels=labs, showfliers=False, patch_artist=True, medianprops=dict(color="white"),
              boxprops=dict(facecolor=nt.OBS, color=nt.OBS), whiskerprops=dict(color=nt.OBS), capprops=dict(color=nt.OBS))
ax[0].set_xscale("log"); ax[0].set_xlabel("car km of the leg (log scale)"); ax[0].invert_yaxis()
c = EST_T.segment.value_counts(); nt.bar(100 * c / c.sum(), "tour segment", "share of legs (%)", ax=ax[1])
c = EST_T.n.clip(upper=6).value_counts().sort_index(); nt.bar(c, "legs of the tour (6 = 6+)", "legs", ax=ax[2])
plt.tight_layout(); plt.show()
items = [("cars0", "no car"), ("carshort", "fewer cars than drivers"), ("lic", "driving licence"), ("age_u18", "under 18"), ("age_65p", "65+"),
         ("peak", "peak period"), ("wash", "home in the WMATA area"), ("rail", "rail tour"), ("stops", "tour with stops"), ("nopath", "no transit path for the leg")]
pd.DataFrame({"share of legs (%)": [100 * EST_T[c].mean() for c, _ in items],
              "transit share if 1 (%)": [100 * (EST_T.y[EST_T[c] == 1] == 1).mean() for c, _ in items]}, index=[l for _, l in items]).round(1)

Leg distance by mode, tour segment, legs per tour, and the person and leg variables with the transit share among the legs that have them.

In [ ]:
# Origin zones of the legs and home zones of the transit tours
L = LEGS["estimation"]; X = X_["estimation"]; Lk = L.loc[X.index[X.chosen_av]]
fig, ax = plt.subplots(1, 2, figsize=(13, 5.6))
nt.zone_map(Lk.oz.value_counts(), "legs (origin zone)", ax=ax[0], log=True)
nt.zone_map(Lk.hz.value_counts(), "legs (home zone of the tour)", ax=ax[1], log=True)
ax[0].set_title("leg origins", fontsize=10); ax[1].set_title("home zones", fontsize=10)
plt.tight_layout(); plt.show()

Origin zones of the legs and home zones of the transit tours.

### Estimation half vs hold-out half

In [ ]:
# Estimation half vs hold-out half
t = nt.half_comparison(EST_T.y.map(LAB), HOLD_T.y.map(LAB), "leg mode")
print(f"records: estimation {len(EST_T):,}, hold-out {len(HOLD_T):,}; largest difference of a share: {t['difference (points)'].abs().max():.1f} points")
t

### 4.1 Variables in the data

Each explanatory variable of the final specification as it enters the utility, interactions included (read from the model's own utility functions: the part of the utility that each coefficient multiplies; coefficients that share a label, e.g. the copies of a term for several purposes, are shown in one panel). Here: each variable by the chosen alternative, estimation half, unweighted. Person and household variables (the same value in every utility they enter) are shown as the mean or share among the records that chose each alternative; attributes of the alternatives (values that differ between alternatives, e.g. cost) as the mean at the chosen alternative against the mean where the alternative is available.

In [ ]:
# every explanatory variable of the specification by chosen alternative
import varplots as vp
MV_ = af.module(MODEL_FOLDER); r_ = MV_.read("estimation"); D_ = r_[0] if isinstance(r_, tuple) else r_
e_ = MV_.formulas() if hasattr(MV_, "formulas") else MV_.loglike(); e_ = e_["loglike"] if isinstance(e_, dict) else e_
U_, AV_, CH_ = vp.find_choice(e_); ALT_NAMES = af.ALTS[MODEL_FOLDER]
VP = vp.by_choice(U_, AV_, CH_, D_, vp.abit_labels(MODEL_FOLDER), ALT_NAMES)
display(Markdown(vp.choice_counts_line(vp.choices(CH_, D_), ALT_NAMES, 'legs')))
vp.by_choice_figures(VP, list(ALT_NAMES.values()), unit='legs')

How to read it: dark grey bars = mean (or share with value 1) among the records that chose the alternative, grey range = 25th-75th percentile, dashed line = all records; for attributes of the alternatives, the light bar is the mean where the alternative is available.

### Data sources

Where each source comes from and where it goes (paths from `manual_estimation/`). *verify*: no download link is recorded in the project. The full download steps are in `INSTRUCTIONS.pdf`.

**2017/18 Regional Travel Survey (Baltimore and Washington regions), household travel survey, weekday travel day** — Baltimore Metropolitan Council (BMC) and the National Capital Region Transportation Planning Board (TPB) at the Metropolitan Washington Council of Governments (MWCOG)
- Where: https://www.baltometro.org/ and https://www.mwcog.org/ (verify: no download page is documented in the project; the data are released on request)
- How: Request the 2017/18 Regional Travel Survey public-use files from BMC (Baltimore region) or MWCOG/TPB (Washington region). The project used one flat file in which every trip row carries its household and person fields (household, person, trip and vehicle tables joined): RTS_raw_flat_data.csv, with the codebook RTS_codebook_data_dictionary.xlsx (sheets Household, Person, Trip, Vehicle). The two cleaned files rts_trips_clean.csv (163,290 trips, deduplicated) and rts_tours.csv (57,627 home-based tours) were made by 01_clean_rts.py of the tour-based MITO project; that script is not on this machine, so these two files are shipped and cannot be rebuilt here. Survey weights: wthhfin, wtperfin, wttrdfin. If you receive separate household / person / trip / vehicle tables instead, join them on household_id, person_id (persno) and tripid to the flat layout of the codebook.
- Put in: 00_shared_data/raw/rts_2017_18 (the project's copies are provided in 00_shared_data/rts_2017_18/raw as .csv.gz)
- Files: `RTS_raw_flat_data.csv`, `RTS_codebook_data_dictionary.xlsx`, `rts_trips_clean.csv`, `rts_tours.csv`

**Maryland Statewide Transportation Model (MSTM) 2016 skims, 1,588 SMZ/RMZ zones** — Maryland Department of Transportation (MDOT SHA), via the project's DAYMAP Maryland set-up (ABIT/input/mito_los/skims)
- Where: https://www.roads.maryland.gov/ (verify: MSTM data are provided by MDOT on request; no download page is documented)
- How: Obtain the MSTM 2016 skims in OMX format (matrix mat1, lookups origins / destinations): traveltime_auto.omx (AM-peak car time, min), distance_auto.omx (car distance, m), bus_tt.omx (bus time, min; 1000 = no path, 5 intrazonal), rail_tt00.omx (rail time, min; 0 = no rail path). convert_mstm_skims.py turns them into the csv.gz files the models read.
- Put in: 00_shared_data/raw/mstm_skims_2016
- Files: `traveltime_auto.omx`, `distance_auto.omx`, `bus_tt.omx`, `rail_tt00.omx`

**MSTM zone system: SMZ/RMZ polygons, zone table, DAYMAP zone attributes** — ABIT/input/maryland/zone_attributes.csv (DAYMAP Maryland set-up)
- Where: https://www.roads.maryland.gov/ (verify: provided with the MSTM)
- How: Zone polygons SMZ_RMZ_02152011inMSTM_EPSG26918.shp (+ .dbf .shx .prj; field id / SMZRMZ, 1,588 zones), zoneSystem.csv (ZoneId, COUNTYFIPS, ...), zone_attributes.csv (id, centroid x/y EPSG:26985, countyfips, region types, bbsr, area_km2;
- Put in: 00_shared_data/raw/mstm_zone_system
- Files: `SMZ_RMZ_02152011inMSTM_EPSG26918.shp/.dbf/.shx/.prj`, `zoneSystem.csv`, `zone_attributes.csv`

**Census 2010 Gazetteer file of census tracts (internal points)** — U.S. Census Bureau
- Where: https://www.census.gov/geographies/reference-files/time-series/geo/gazetteer-files.html (verify: the project copy I695_pricing/data/census/Gaz_tracts_national.txt has no download log)
- How: 2010 Gazetteer, Census Tracts, national file Gaz_tracts_national.txt (GEOID, INTPTLAT, INTPTLONG). Used by survey_population.tract_zone: zone of each tract's internal point.
- Put in: 00_shared_data/raw/census_gazetteer_2010
- Files: `Gaz_tracts_national.txt`

**MTA Maryland GTFS static feeds: local bus, light rail, metro, MARC, commuter bus** — Maryland Transit Administration
- Where: https://feeds.mta.maryland.gov/gtfs/<feed> for feed = local-bus, light-rail, metro, marc, commuter-bus (listed on https://www.mta.maryland.gov/developer-resources)
- How: Run 00_shared_prep/code/fetch_gtfs.sh (curl, one zip per feed, download log). The project used the feeds of 2026-10-21 (local bus valid 2026-08-21 to 2027-02-06).
- Put in: 00_shared_data/raw/gtfs/mta_gtfs
- Files: `mta_local-bus.zip`, `mta_light-rail.zip`, `mta_metro.zip`, `mta_marc.zip`, `mta_commuter-bus.zip`

**GTFS of the 13 regional operators (with the 5 MTA feeds the 18 feeds merged by merge_gtfs_full.py)** — each operator; Mobility Database mirror
- Where: charm_city_circulator: https://transportation.baltimorecity.gov/files/cccgtfs824zip (mirror https://files.mobilitydatabase.org/tld-288_1/latest.zip ; https://files.mobilitydatabase.org/ntd-30201/latest.zip ; https://files.mobilitydatabase.org/mdb-493/latest.zip); rta_central_maryland: http://data.trilliumtransit.com/gtfs/centralmarylandrta-md-us/centralmarylandrta-md-us.zip (mirror https://files.mobilitydatabase.org/mdb-489/latest.zip); anne_arundel_transit: https://www.aacounty.org/gtfs (mirror https://files.mobilitydatabase.org/ntd-30129/latest.zip); annapolis_transit: https://www.arcgis.com/sharing/rest/content/items/c307923d57ab4871aff79491b4a1bfd3/data (mirror https://files.mobilitydatabase.org/mdb-2285/latest.zip); harford_transit_link: https://files.mobilitydatabase.org/mdb-501/latest.zip (mirror https://github.com/mobilityequity/maryland-local-gtfs/raw/master/Harford_GTFS.zip); carroll_transit: https://files.mobilitydatabase.org/mdb-500/latest.zip (mirror https://github.com/mobilityequity/maryland-local-gtfs/raw/master/Carroll_GTFS.zip); wmata_rail: https://api.wmata.com/gtfs/rail-gtfs-static.zip (API key required) (mirror https://files.mobilitydatabase.org/mdb-1847/latest.zip); wmata_bus: https://api.wmata.com/gtfs/bus-gtfs-static.zip (API key required) (mirror https://files.mobilitydatabase.org/mdb-1846/latest.zip); ride_on: https://www.montgomerycountymd.gov/DOT-Transit/Resources/Files/GTFS/RideOnGTFS.zip (mirror https://files.mobilitydatabase.org/mdb-488/latest.zip); thebus_pg: https://data.trilliumtransit.com/gtfs/princegeorgescounty-md-us/princegeorgescounty-md-us.zip (mirror https://files.mobilitydatabase.org/mdb-477/latest.zip); amtrak: https://content.amtrak.com/content/gtfs/GTFS.zip (mirror https://files.mobilitydatabase.org/mdb-11/latest.zip); harbor_connector: http://data.trilliumtransit.com/gtfs/harborconnector-md-us/harborconnector-md-us.zip (mirror https://files.mobilitydatabase.org/mdb-618/latest.zip); towson_loop: https://passio3.com/towson/passioTransit/gtfs/google_transit.zip (mirror https://files.mobilitydatabase.org/mdb-1784/latest.zip)
- How: Run 00_shared_prep/code/fetch_gtfs_regional.sh (official URL first, then the Mobility Database mirror; download log with sha256). The script also fetches the Bethesda Circulator (https://files.mobilitydatabase.org/mdb-231/latest.zip), which merge_gtfs_full.py does not use.
- Put in: 00_shared_data/raw/gtfs/<agency>
- Files: `charm_city_circulator/charm_city_circulator_gtfs.zip`, `rta_central_maryland/rta_central_maryland_gtfs.zip`, `anne_arundel_transit/anne_arundel_transit_gtfs.zip`, `annapolis_transit/annapolis_transit_gtfs.zip`, `harford_transit_link/harford_transit_link_gtfs.zip`, `carroll_transit/carroll_transit_gtfs.zip`, `wmata_rail/wmata_rail_gtfs.zip`, `wmata_bus/wmata_bus_gtfs.zip`, `ride_on/ride_on_gtfs.zip`, `thebus_pg/thebus_pg_gtfs.zip`, `amtrak/amtrak_gtfs.zip`, `harbor_connector/harbor_connector_gtfs.zip`, `towson_loop/towson_loop_gtfs.zip`

## 5. Model specification

*The utility functions, read from the model script; nothing is re-specified here.*

The folder's script `model/estimate_transit_leg_mode.py`, imported as it is. Its utility functions:

In [ ]:
# Model specification
sys.path.insert(0, os.path.join(FOLDER, "model")); sys.modules.pop("estimate_transit_leg_mode", None)   # the model folder's script (input/code may hold a namesake)
import estimate_transit_leg_mode as M
src = inspect.getsource(M)
print(src[:src.find("def read(")])

In [ ]:
# Model specification
r = M.read("estimation")
df = r[0] if isinstance(r, tuple) else r
print(f"estimation table as read by the script: {df.shape[0]:,} rows, {df.shape[1]} columns")

## 6. Correlation of the explanatory variables

*Correlation of the explanatory variables. Look for: |r| of 0.7 or more and VIF of 10 or more; these make single coefficients less precise.*

|r| ≥ 0.7 and VIF ≥ 10 are listed. Interaction terms are correlated with their main effects by construction.

In [ ]:
# Correlation of the explanatory variables
f = M.formulas() if hasattr(M, "formulas") else M.loglike()
expr = f["loglike"] if isinstance(f, dict) else f
used = [c for c in nt.model_variables(expr) if c in df and not c.startswith("av_") and c not in ("y", "wesml", "w", "weight")]
Xu = nt.explanatory(df[used])
print(f"{len(used)} data columns enter the utilities")
Cu = nt.corr_heatmap(Xu, "Variables of the utilities")
display(nt.high_pairs(Cu)); display(nt.vif(Xu))

## 7. Log-likelihood at the reference coefficients

*Log-likelihood at the article coefficients, without estimation. Look for: the two numbers should be equal (to rounding).*

No estimation: the script's `evaluate("estimation")`, i.e. the log-likelihood of the estimation half at the full-precision coefficients
of `model/reference_estimates.csv`. Target: LL(beta) of that file, -3421.4993248855308 (article -3421.4993;
N = 6773, K = 35).

In [ ]:
# Log-likelihood at the reference coefficients
LL_BETA = -3421.4993248855308
def evaluate(half):
    here = os.getcwd()
    try:
        return M.evaluate(half, None) if "at" in inspect.signature(M.evaluate).parameters else M.evaluate(half)
    finally:
        os.chdir(here)

if RUN_CHECK:
    t0 = time.time(); ll_ref = evaluate("estimation")
    print(f"{(time.time() - t0) / 60:.1f} min")
    if isinstance(ll_ref, (int, float)) and LL_BETA is not None:
        print(f"LL at the reference coefficients {ll_ref:,.4f}; LL(beta) of reference_estimates.csv {LL_BETA:,.4f}; difference {ll_ref - LL_BETA:+.4f}")
else:
    print("skipped (RUN_CHECK = False)")

## 8. Biogeme estimation

*Biogeme estimation, only with RUN_ESTIMATION. Look for: convergence and the run time.*

The script's `estimate()`: Biogeme with its defaults (`output/biogeme.toml`), robust standard errors. Report in
`output/transit_leg_mode.html` and `.yaml`. Not run here (`RUN_ESTIMATION = False`); a result in `output/` is read only if it is
newer than `model/reference_estimates.csv`.

In [ ]:
# Biogeme estimation
if RUN_ESTIMATION:
    here = os.getcwd(); t0 = time.time()
    try:
        M.estimate(OUT)
    finally:
        os.chdir(here)
    print(f"estimation time {time.time()-t0:.0f} s")
ref_time = os.path.getmtime(os.path.join(FOLDER, "model", "reference_estimates.csv"))
y = [f for f in sorted(glob.glob(os.path.join(OUT, "*.yaml")), key=os.path.getmtime) if "eval" not in os.path.basename(f) and os.path.getmtime(f) > ref_time]
results = EstimationResults.from_yaml_file(y[-1]) if y else None
print(f"results read from {os.path.basename(y[-1])}" if y else "no current result in output/: the estimate tables of section 9 are skipped")

## 9. Results

*The estimates and what they mean. Look for: coefficients equal to the article within rounding, and the readings below.*

### Estimates in the article format
Coefficient (stars: *** p < 0.01, ** p < 0.05, * p < 0.10, as in the article), robust s.e., p.

In [ ]:
# Results
if results is not None:
    est_t = nt.estimates_table(results)
    display(nt.article_format(est_t))

### What the coefficients say

One plain line per term (article estimates of `model/reference_estimates.csv`) and the coefficient plot.

In [ ]:
# plain readings of the coefficients and the coefficient plot (95 % robust intervals, grouped by block)
rows = af.coef_rows(MODEL_FOLDER)
display(rd.abit_readings(rows).style.hide(axis="index").set_properties(**{"text-align": "left"}))
fg.coef_figure(rows); plt.show()

How to read it: a point right of zero makes the alternative more likely; an interval that crosses zero is not significant; colours are the alternatives or purposes a term belongs to.

### Goodness of fit

In [ ]:
# Goodness of fit
if results is not None:
    display(nt.fit_statistics(results))

### Comparison with the article (`model/reference_estimates.csv`)

In [ ]:
# Comparison with the article (model/reference_estimates.csv)
if results is not None:
    REF = os.path.join(FOLDER, "model", "reference_estimates.csv")
    cmp = nt.compare(est_t, REF)
    cmp.to_csv(os.path.join(OUT, "comparison_with_article_notebook.csv"), index=False)
    print(f"{int(cmp.matches_article.sum())} of {len(cmp)} parameters match the article (coefficient and robust s.e. within the 4th-decimal rounding); "
          f"largest |difference|: coefficient {cmp.diff_coef.abs().max():.5f}, s.e. {cmp.diff_se.abs().max():.5f}")
    display(cmp.round({"coefficient": 4, "robust_se": 4, "p": 4, "diff_coef": 5, "diff_se": 5}))
    stats = nt.article_statistics(REF)
    if len(stats):
        stats["Biogeme"] = stats["statistic"].map({"LL(beta)": results.final_loglikelihood, "K": results.number_of_free_parameters, "N": 6773})
        display(stats)
    nt.plot_against_article(cmp, RUN.get("model", NAME))

### Correlation of the estimates
Robust correlation matrix of the estimated parameters; pairs with |r| ≥ 0.7 listed.

In [ ]:
# Correlation of the estimates
if results is not None:
    display(nt.parameter_correlation(results, "Robust correlation of the estimates"))

## 10. Hold-out sample

*Fit on the hold-out records, which the estimation did not use. Look for: a log-likelihood per record close to that of the estimation sample, and the maps.*

Log-likelihood of the hold-out half at the reference coefficients (`evaluate("holdout")`).

In [ ]:
# Hold-out sample
if RUN_HOLDOUT:
    ll_hold = evaluate("holdout")
else:
    print("skipped (RUN_HOLDOUT = False)")

### Hold-out shares

Observed and model shares of the hold-out half (model: mean of the probabilities at the reference coefficients, simulated with this script's own log-likelihood).

In [ ]:
# observed against model shares on the hold-out half, overall and for one segment
t1, (title, t2, xl), n_hold = af.holdout_tables(MODEL_FOLDER)
t2 = t2[(t2.observed > 0.01) | (t2.model > 0.01)]
A = af.ALTS[MODEL_FOLDER]
fg.obs_model_figure([("All" if len(A) > 2 else t1.index[0], t1, "", "Hold-out records (%)"), (title, t2, xl, f"{t1.index[0] if len(A) == 2 else A[1]} (%)")]); plt.show()
print(f"hold-out records: {n_hold:,}")

How to read it: dark grey observed, orange model; equal bars mean the model reproduces the hold-out shares overall and within the segment. The hold-out half was also used in the specification search.

## Glossary

- **DAYMAP**: Daily Activity Model for Agent Plans: the activity-based model of the MILES platform, built on the ABIT code base (Activity-Based Incremental Transport model).
- **reference alternative**: The alternative whose utility is fixed at 0; the coefficients of the others are differences against it.
- **log-likelihood**: Sum over decision makers of ln(probability of the observed choice). Closer to 0 is better.
- **hold-out**: The 20 % (residential, cars) or half (survey models) of the records not used for estimation. It was also used in the specification search, so its fit is not a fully independent test.
- **robust s.e.**: Standard error from the sandwich formula, valid when the model is not exactly right.
- **VIF**: Variance inflation factor: how much the correlation with other variables inflates a coefficient's variance; 10 or more is high.